<div class="alert alert-block alert-info">
    <h1>Análisis de Series Temporales</h1>
    <h3>Clase 7 - PYtimeTK - Anomalias </h3>
    <h3>Ejercicio 3</h3>
        <p>Docentes: Sebastian Calcagno, Rodrigo Del Rosso, Drago Braian<p>
</div>

Modeltime: La extensión de Tidymodels para el modelado de series temporales  

El marco de trabajo para la predicción de series temporales, compatible con el ecosistema de 'tidymodels'. Incluye modelos ARIMA, suavizado exponencial y otros modelos de series temporales de los paquetes 'forecast' y 'prophet'.

https://business-science.github.io/modeltime/

PyTimeTK nos permite trabajar con series temporales utilizando una sintaxis bastante parecida a Pandas. En este ejemplo vemos como resumir informacion por tiempo y, principalmente, como automatizar la deteccion y visualizacion de comportamientos anomalos

**Objetivo del Ejercicio**: Introducción a la librería Model Time de R  

1- Herramientas Generales  
2- Ejemplo aplicado

# **PYtimeTK**

https://business-science.github.io/pytimetk/guides/06_anomalize.html

In [ ]:
!pip install pytimetk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.3/985.3 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.5 MB/s eta 0:00:00
  Created wheel for supersmoother: filename=supersmoother-0.4-py3-none-any.whl size=16146 sha256=85152fe87bcccbccf1fcd7520b4439dff6c791805e98543d8a2c2308d2cf6913
  Stored in directory: /root/.cache/pip/wheels/ce/54/3b/3a6a20584d667a3a898b09ead38658bc5b8b3f68cb2875fb6d
Successfully built supersmoother
  Attempting uninstall: dill
    Found existing installation: dill 0.3.7
    Uninstalling dill-0.3.7:

In [ ]:
import pytimetk as tk
import pandas as pd

df = tk.load_dataset('bike_sales_sample')
df['order_date'] = pd.to_datetime(df['order_date'])

df

,order_id,order_line,order_date,quantity,price,total_price,model,category_1,category_2,frame_material,bikeshop_name,city,state
0,1,1,2011-01-07,1,6070,6070,Jekyll Carbon 2,Mountain,Over Mountain,Carbon,Ithaca Mountain Climbers,Ithaca,NY
1,1,2,2011-01-07,1,5970,5970,Trigger Carbon 2,Mountain,Over Mountain,Carbon,Ithaca Mountain Climbers,Ithaca,NY
2,2,1,2011-01-10,1,2770,2770,Beast of the East 1,Mountain,Trail,Aluminum,Kansas City 29ers,Kansas City,KS
3,2,2,2011-01-10,1,5970,5970,Trigger Carbon 2,Mountain,Over Mountain,Carbon,Kansas City 29ers,Kansas City,KS
4,3,1,2011-01-10,1,10660,10660,Supersix Evo Hi-Mod Team,Road,Elite Road,Carbon,Louisville Race Equipment,Louisville,KY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2461,321,3,2011-12-22,1,1410,1410,CAAD8 105,Road,Elite Road,Aluminum,Miami Race Equipment,Miami,FL
2462,322,1,2011-12-28,1,1250,1250,Synapse Disc Tiagra,Road,Endurance Road,Aluminum,Phoenix Bi-peds,Phoenix,AZ
2463,322,2,2011-12-28,1,2660,2660,Bad Habit 2,Mountain,Trail,Aluminum,Phoenix Bi-peds,Phoenix,AZ
2464,322,3,2011-12-28,1,2340,2340,F-Si 1,Mountain,Cross Country Race,Aluminum,Phoenix Bi-peds,Phoenix,AZ


Utilizando summarize_by_time() para un análisis de ventas: El resultado es el ingreso total por bicicletas de montaña y de carretera por mes.

In [ ]:
summary_category_1_df = df \
    .groupby("category_1") \
    .summarize_by_time(
        date_column  = 'order_date',
        value_column = 'total_price',
        freq         = "MS",
        agg_func     = 'sum',
        wide_format  = False
    )

summary_category_1_df.head()

,category_1,order_date,total_price
0,Mountain,2011-01-01,221490
1,Mountain,2011-02-01,660555
2,Mountain,2011-03-01,358855
3,Mountain,2011-04-01,1075975
4,Mountain,2011-05-01,450440


Visualización de patrones de ventas

In [ ]:
summary_category_1_df \
    .groupby('category_1') \
    .plot_timeseries(
        date_column  = 'order_date',
        value_column = 'total_price',
        smooth_frac  = 0.8
    )

**Anomaly Detection**

**m4_monthly**: muestra de 4 series de tiempo de frecuencia mensual tomadas de la M4 competition.

Las competiciones de Makridakis son una serie de concursos abiertos para evaluar y comparar la precisión de diferentes métodos de predicción de series temporales.

A continuación cargamos los datos, seleccionamos in ID de interés y visualizamos la serie:

In [ ]:

import pytimetk as tk
import pandas as pd
import numpy as np


m4_daily_df = tk.load_dataset('m4_daily', parse_dates = ['date'])

In [ ]:

df = (
    m4_daily_df
        .query("id == 'D10'")
        .query("date.dt.year == 2015")
)

In [ ]:
df.head()

,id,date,value
182,D10,2015-01-01,2351.0
183,D10,2015-01-02,2302.7
184,D10,2015-01-03,2300.7
185,D10,2015-01-04,2341.2
186,D10,2015-01-05,2351.4


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 365 entries, 182 to 546
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   id      365 non-null    object        
 1   date    365 non-null    datetime64[ns]
 2   value   365 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 11.4+ KB


In [ ]:
# Ploteo
tk.plot_timeseries(
    data         = df,
    date_column  = 'date',
    value_column = 'value'
)

Ahora, como primer paso, se realiza una descomposición estacional y se generan remainders utilizando anomalize()

**iqr_alpha** controla el umbral para detectar valores atípicos. Es el nivel de significancia utilizado en el método del rango intercuartil (IQR) para la detección de valores atípicos. El valor predeterminado es 0,05, que corresponde a un nivel de significancia del 5%. Un nivel de significancia más bajo dará como resultado un umbral más alto, lo que significa que se detectarán menos valores atípicos. Un nivel de significancia más alto dará como resultado un umbral más bajo, lo que significa que se detectarán más valores atípicos.


La funcion primero analiza la estructura de la serie, separa el comportamiento esperado y despues identifica los puntos que se alejan demasiado de ese comportamiento. Ademas genera una version limpia de la serie reemplazando esos valores extremos

In [ ]:
# Anomalize
anomalize_df = tk.anomalize(
    data          = df,
    date_column   = 'date',
    value_column  = 'value',
    period        = 7,
    iqr_alpha     = 0.05, # using the default
    clean_alpha   = 0.75, # using the default
    clean         = "min_max"
)

anomalize_df.glimpse()

<class 'pandas.core.frame.DataFrame'>: 365 rows of 12 columns
date:               datetime64[ns]    [Timestamp('2015-01-01 00:00:00'), ...
observed:           float64           [2351.0, 2302.7, 2300.7, 2341.2, 2 ...
seasonal:           float64           [14.163009085035966, -17.341946034 ...
seasadj:            float64           [2336.836990914964, 2320.041946034 ...
trend:              float64           [2323.9003178512266, 2322.99646033 ...
remainder:          float64           [12.936673063737544, -2.9545142990 ...
anomaly:            object            ['No', 'No', 'No', 'No', 'No', 'No ...
anomaly_score:      float64           [19.42215274680052, 35.31334010958 ...
anomaly_direction:  int64             [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  ...
recomposed_l1:      float64           [2179.8604039090906, 2147.45159127 ...
recomposed_l2:      float64           [2560.9839015845105, 2528.57508894 ...
observed_clean:     float64           [2351.0, 2302.7, 2300.7, 2341.2, 2 ...


Visualizamos la seaonal decomposition para ver su representación:

In [ ]:
# Gráfico de descomposición estacional  --> estamos buscando principalmente un patron semanal.
tk.plot_anomalies_decomp(
    data        = anomalize_df,
    date_column = 'date',
    engine      = 'plotly',
    title       = 'Seasonal Decomposition'
)

Arriba tenemos la serie original. PyTimeTK la separa en estacionalidad, tendencia y residuo. La tendencia nos muestra una fuerte caida hacia mitad de año y una recuperacion posterior. Para detectar anomalias nos interesa especialmente el ultimo grafico, el remainder. Cuando el residuo se aleja mucho de cero significa que ocurrio algo que la tendencia y la estacionalidad no pueden explicar. En este caso, el periodo mas llamativo aparece entre julio y agosto. Los valores reales fueron bastante menores a los que se esperaban segun el comportamiento normal de la serie

Visualizamos las anomalías ustilizando tk.plot_anomalies();

In [ ]:
# Plot anomalies
tk.plot_anomalies(
    data        = anomalize_df,
    date_column = 'date',
    engine      = 'plotly',
    title       = 'Plot Anomaly Bands'
)

Finalmente observamos la gráfica con los datos limpios utilizando plot_anomalies_cleaned()

In [ ]:
# Plot cleaned anomalies
tk.plot_anomalies_cleaned(
    data        = anomalize_df,
    date_column = 'date'
)